# Behavior Cloning Dataset Generation

This notebook generates a trade log for a specific high-performing strategy to be used as an expert dataset for Behavior Cloning.

**Strategy:** 1hr Long, No Stop Loss.

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick
from tqdm.auto import tqdm

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

/home/patrics/repos/ElegantRL/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- CONFIGURATION ---
DATA_DIR = "/opt/rws/repos/RWS_LightGBM/notebooks/experiments"
FILENAME = "2001-2012_val_2015_merged_predictions_ohlcv.parquet"
FILE_PATH = os.path.join(DATA_DIR, FILENAME)

HORIZON = '1h'
HORIZON_MINS = 60
TRADE_DIRECTION = 'LONG'
THRESHOLD = 0.6
MAX_ACTIVE_TRADES = 10
CAPITAL_PER_TRADE = 10000
COST_BPS = 2.0 
START_CAPITAL = 100000

# Time Filters
START_TIME = "09:45"
END_TIME = "15:30"
ALLOW_OVERNIGHT = False
EOD_TIME = "16:00"

# Output
OUTPUT_CSV = "BC_expert_trades_1h_long.csv"

In [3]:
# --- LOAD DATA ---
print(f"Loading {FILE_PATH}...")
try:
    df = pd.read_parquet(FILE_PATH)
    print("Data loaded successfully.")
except FileNotFoundError:
    print(f"Error: File not found at {FILE_PATH}. Please check the path.")
    # Attempting to find it in the current workspace if it fails
    if os.path.exists(FILENAME):
        df = pd.read_parquet(FILENAME)
        print("Data loaded from current directory.")
    else:
        raise

# Pre-processing
if 'timestamp' not in df.columns and 'date' in df.columns:
    df['timestamp'] = pd.to_datetime(df['date'])
elif 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'])

sim_df = df.sort_values('timestamp').copy()
print(f"Total rows: {len(sim_df)}")
print(f"Date range: {sim_df['timestamp'].min()} to {sim_df['timestamp'].max()}")

Loading /opt/rws/repos/RWS_LightGBM/notebooks/experiments/2001-2012_val_2015_merged_predictions_ohlcv.parquet...
Data loaded successfully.
Total rows: 4247679
Date range: 2015-01-02 09:30:00 to 2015-12-31 16:00:00


In [4]:
def calculate_performance_stats(trades_df, equity_curve_daily):
    """Generates professional financial metrics from trade log."""
    if trades_df.empty: return {}
    
    wins = trades_df[trades_df['net_ret'] > 0]
    losses = trades_df[trades_df['net_ret'] <= 0]
    
    n_trades = len(trades_df)
    win_rate = len(wins) / n_trades if n_trades > 0 else 0
    avg_win = wins['pnl'].mean() if not wins.empty else 0
    avg_loss = abs(losses['pnl'].mean()) if not losses.empty else 0
    
    gross_profit = wins['pnl'].sum()
    gross_loss = abs(losses['pnl'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss != 0 else np.inf
    expectancy = (win_rate * avg_win) - ((1 - win_rate) * avg_loss)
    
    sqn = np.sqrt(n_trades) * (trades_df['pnl'].mean() / trades_df['pnl'].std()) if trades_df['pnl'].std() > 0 else 0
    
    daily_rets = equity_curve_daily.diff().fillna(0) / START_CAPITAL 
    sharpe = (daily_rets.mean() / daily_rets.std()) * np.sqrt(252) if daily_rets.std() > 0 else 0
    
    rolling_max = equity_curve_daily.cummax()
    max_dd_dollar = (equity_curve_daily - rolling_max).min()
    
    return {
        "Total Net Profit": trades_df['pnl'].sum(),
        "Total Trades": n_trades,
        "Win Rate": win_rate,
        "Profit Factor": profit_factor,
        "Avg Win": avg_win,
        "Avg Loss": avg_loss,
        "Expectancy ($)": expectancy,
        "SQN": sqn,
        "Sharpe Ratio": sharpe,
        "Max Drawdown ($)": max_dd_dollar
    }

In [5]:
def run_slot_simulation(data, max_slots, threshold, horizon_m, direction, start_str, end_str, overnight_allowed, eod_str):
    active_trades = [] 
    completed_trades = []
    
    prob_col = f'prob_up_{HORIZON}' if direction == 'LONG' else f'prob_down_{HORIZON}'
    ret_col = f'target_reg_{HORIZON}_logret'
    ret_mult = 1.0 if direction == 'LONG' else -1.0
    
    start_t = pd.to_datetime(start_str).time()
    end_t = pd.to_datetime(end_str).time()
    eod_t = pd.to_datetime(eod_str).time()
    
    # Optimize: Pre-filter relevant columns
    data = data[['timestamp', 'ticker', prob_col, ret_col]].copy()
    
    timestamps = data['timestamp'].unique()
    data_by_time = dict(tuple(data.groupby('timestamp')))
    concurrency_log = [] 
    
    print(f"🔄 Simulating {len(timestamps)} time steps ({direction} logic)...")
    
    for current_time in tqdm(timestamps, desc="Backtesting"):
        # 1. EXITS
        active_trades = [t for t in active_trades if t['exit_time'] > current_time]
        
        current_slots_filled = len(active_trades)
        slots_available = max_slots - current_slots_filled
        
        # 2. EVALUATE TIME CONSTRAINTS
        curr_t = current_time.time()
        can_enter = False
        
        if start_t <= curr_t <= end_t:
            can_enter = True
            if not overnight_allowed:
                proposed_exit = current_time + pd.Timedelta(minutes=horizon_m)
                if proposed_exit.date() != current_time.date() or proposed_exit.time() > eod_t:
                    can_enter = False
        
        # 3. ENTRIES
        if slots_available > 0 and can_enter:
            step_data = data_by_time.get(current_time)
            if step_data is not None:
                candidates = step_data[step_data[prob_col] > threshold]
                if not candidates.empty:
                    # Sort by Conviction
                    candidates = candidates.sort_values(prob_col, ascending=False)
                    to_enter = candidates.head(slots_available)
                    
                    for _, row in to_enter.iterrows():
                        trade = {
                            'entry_time': current_time,
                            'exit_time': current_time + pd.Timedelta(minutes=horizon_m),
                            'ticker': row['ticker'],
                            'prob': row[prob_col],
                            'raw_ret': row[ret_col] * ret_mult
                        }
                        active_trades.append(trade)
                        completed_trades.append(trade)
        
        concurrency_log.append({'timestamp': current_time, 'active_count': len(active_trades)})
        
    return pd.DataFrame(completed_trades), pd.DataFrame(concurrency_log)

In [6]:
# --- RUN SIMULATION ---
trades_log, concurrency_df = run_slot_simulation(
    sim_df, MAX_ACTIVE_TRADES, THRESHOLD, HORIZON_MINS, 
    TRADE_DIRECTION, START_TIME, END_TIME, ALLOW_OVERNIGHT, EOD_TIME
)

if not trades_log.empty:
    # Economics
    cost_decimal = COST_BPS / 10000.0
    trades_log['net_ret'] = trades_log['raw_ret'] - cost_decimal
    trades_log['pnl'] = trades_log['net_ret'] * CAPITAL_PER_TRADE
    
    # Curve
    daily_pnl = trades_log.set_index('exit_time')['pnl'].resample('D').sum().fillna(0)
    equity_curve = daily_pnl.cumsum()
    
    # Stats
    stats = calculate_performance_stats(trades_log, equity_curve)
    
    print("\n" + "="*45)
    print(f"📊 STRATEGY PERFORMANCE REPORT")
    print("="*45)
    for label, val in stats.items():
        if isinstance(val, float):
            print(f"{label:<25} {val:,.4f}")
        else:
            print(f"{label:<25} {val}")
    print("="*45)
    
    # Save to CSV
    trades_log.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ Trade log saved to {OUTPUT_CSV}")
else:
    print("❌ No trades generated. Try lowering the threshold.")

🔄 Simulating 19855 time steps (LONG logic)...


Backtesting: 100%|██████████| 19855/19855 [00:02<00:00, 8137.99it/s]


📊 STRATEGY PERFORMANCE REPORT
Total Net Profit          33,717.5661
Total Trades              3092
Win Rate                  0.5417
Profit Factor             1.3482
Avg Win                   77.9419
Avg Loss                  68.3382
Expectancy ($)            10.9048
SQN                       5.6574
Sharpe Ratio              2.3515
Max Drawdown ($)          -5,056.7158

✅ Trade log saved to BC_expert_trades_1h_long.csv


In [7]:
trades_log

,entry_time,exit_time,ticker,prob,raw_ret,net_ret,pnl
0,2015-01-02 10:35:00,2015-01-02 11:35:00,KSS,0.625553,-0.002841,-0.003041,-30.4115
1,2015-01-02 10:40:00,2015-01-02 11:40:00,KSS,0.755218,0.003349,0.003149,31.4896
2,2015-01-02 10:40:00,2015-01-02 11:40:00,LVS,0.696471,0.008250,0.008050,80.4969
3,2015-01-02 10:40:00,2015-01-02 11:40:00,HLT,0.667773,0.003713,0.003513,35.1317
4,2015-01-02 10:40:00,2015-01-02 11:40:00,DHI,0.662513,-0.001615,-0.001815,-18.1486
...,...,...,...,...,...,...,...
3087,2015-12-31 11:05:00,2015-12-31 12:05:00,SO,0.646948,0.004502,0.004302,43.0210
3088,2015-12-31 11:05:00,2015-12-31 12:05:00,D,0.601133,0.003414,0.003214,32.1424
3089,2015-12-31 11:10:00,2015-12-31 12:10:00,SO,0.673540,0.005709,0.005509,55.0854
3090,2015-12-31 11:10:00,2015-12-31 12:10:00,DUK,0.663785,0.007175,0.006975,69.7454


In [8]:
pos_ret = trades_log[trades_log['net_ret'] > 0]
pos_ret

,entry_time,exit_time,ticker,prob,raw_ret,net_ret,pnl
1,2015-01-02 10:40:00,2015-01-02 11:40:00,KSS,0.755218,0.003349,0.003149,31.4896
2,2015-01-02 10:40:00,2015-01-02 11:40:00,LVS,0.696471,0.008250,0.008050,80.4969
3,2015-01-02 10:40:00,2015-01-02 11:40:00,HLT,0.667773,0.003713,0.003513,35.1317
5,2015-01-02 10:40:00,2015-01-02 11:40:00,DE,0.655795,0.002627,0.002427,24.2692
7,2015-01-02 10:40:00,2015-01-02 11:40:00,GM,0.648468,0.003914,0.003714,37.1390
...,...,...,...,...,...,...,...
3087,2015-12-31 11:05:00,2015-12-31 12:05:00,SO,0.646948,0.004502,0.004302,43.0210
3088,2015-12-31 11:05:00,2015-12-31 12:05:00,D,0.601133,0.003414,0.003214,32.1424
3089,2015-12-31 11:10:00,2015-12-31 12:10:00,SO,0.673540,0.005709,0.005509,55.0854
3090,2015-12-31 11:10:00,2015-12-31 12:10:00,DUK,0.663785,0.007175,0.006975,69.7454


In [9]:
pos_ret.describe().T

,count,mean,min,25%,50%,75%,max,std
entry_time,1675,2015-07-08 07:33:03.582089728,2015-01-02 10:40:00,2015-04-10 10:37:30,2015-07-22 14:55:00,2015-10-15 10:20:00,2015-12-31 11:10:00,NaN
exit_time,1675,2015-07-08 08:33:03.582089728,2015-01-02 11:40:00,2015-04-10 11:37:30,2015-07-22 15:55:00,2015-10-15 11:20:00,2015-12-31 12:10:00,NaN
prob,1675.0,0.656003,0.600022,0.615323,0.638916,0.680534,0.937587,0.053277
raw_ret,1675.0,0.007994,0.000203,0.002532,0.005529,0.01053,0.139685,0.008356
net_ret,1675.0,0.007794,0.000003,0.002332,0.005329,0.01033,0.139485,0.008356
pnl,1675.0,77.941948,0.02956,23.32455,53.2861,103.3045,1394.85,83.563256
